# Sparse chemistry

Build the original sparse PREPARE and SELECT fixtures, inspect their
signatures and source cost graphs, and bind a native walk step.

Adapted from Qualtran v0.7.0. Copyright 2024 Google LLC.
Licensed under Apache-2.0.

In [ ]:
from IPython.display import SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.sparse_chemistry_resources import (
    sparse_integral_fixture, sparse_prepare_graph, sparse_select_graph,
)
from flagquantum.algorithms.primitives.multiplexers.select_sparse import SelectSparse
from flagquantum.algorithms.primitives.sparse_chemistry_walk import SparseChemistryWalk
from flagquantum.compiler._component_trace.call_graph import modeled_call_graph
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.drawer.call_graph import export_modeled_call_graph_dot, render_call_graph_svg

def show_signature(signature, model):
    display(SVG(field_signature_svg([(model.name, signature, {})])))

def show_calls(model):
    display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1))))
    leaves = modeled_call_graph(model, max_depth=1).leaf_counts
    for component, count in leaves.items():
        print(component.name, count)
    print(estimate_component_resources(model))

## PREPARE

Use modified one-electron integrals and a four-index chemists' tensor.
The original six-spin-orbital fixture uses seeded integrals and four
bits of alias precision. Selection words persist through PREPARE.
Selected and retained QROAM cells carry the alias outputs.

### Example

In [ ]:
walk = SparseChemistryWalk.from_integrals(
    *sparse_integral_fixture(3), probability_bits=4
)
prepare = walk.prepare
prepare_model = sparse_prepare_graph(prepare)
print("Coefficient one-norm", walk.alpha)

### Register signature

In [ ]:
show_signature(prepare.signature, prepare_model)

### Call graph

In [ ]:
show_calls(prepare_model)

## SELECT

The original four-spin-orbital example uses bounded spatial indices,
spin bits, and the one-body flag. The flag controls the first pair of
Majoranas and carries the S phase. The second pair acts unconditionally.

### Example

In [ ]:
select_width = 4
select_signature = SelectSparse.source_signature(select_width)
select_model = sparse_select_graph(select_width)

### Register signature

In [ ]:
show_signature(select_signature, select_model)

### Call graph

In [ ]:
show_calls(select_model)

### Bind a native walk step

The factory accepts modified and chemists' integrals. It retains the
complete PREPARE signal, including the lookup outputs, when it reflects.
Use the same wires for the forward and adjoint walks. The locked resource
model describes a narrower reflection and prices an abstract QROAM
eraser; its estimates have a separate scope from these emitted counts.

In [ ]:
from collections import Counter

from flagquantum.compiler.registers.layout.partition import partition_register
from flagquantum.dynamic import DynamicCircuit

small = SparseChemistryWalk.from_integrals(
    [[1.0]], [[[[-0.5]]]], probability_bits=1, log_block_size=0
)
logical = sum(bits for _, bits in small.layout)
work, slots = small.requirements()
ports = partition_register(tuple(range(logical)), small.layout)
op = small.make_on(ports, workspace=tuple(range(logical, logical + work)),
                   classical_bits=tuple(range(slots)))
circuit = DynamicCircuit(logical + work).append(op)
emitted = Counter(gate.name for gate in circuit.to_ir().instructions)
print("Emitted measurements", emitted["measure"])
print("Emitted T and T adjoints", emitted["t"] + emitted["tdg"])
print("Reusable work wires", work)